In [ ]:
# Standard library imports
import json
import os
import pickle
import random
import warnings
from collections import Counter, defaultdict
from pathlib import Path

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm import tqdm

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    """Set random seeds for reproducibility.
    
    Args:
        seed: Random seed value (default: 42)
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

print(f"PyTorch: {torch.__version__}")
print(f"TIMM: {timm.__version__}")


# CONFIG - STRONGER SUPCON SETTING


In [ ]:
class Config:
    """Configuration class for model training and dataset settings."""
    BASE_PATH = "/kaggle/input/stages/Stages"
    OUTPUT_BASE = "/kaggle/working"
    
    MODEL_NAME = "convnextv2_tiny.fcmae_ft_in22k_in1k"
    IMG_SIZE = 256
    EMBEDDING_DIM = 512
    
    BATCH_SIZE = 64
    EPOCHS = 100
    LEARNING_RATE = 1e-4
    NUM_WORKERS = 4
    WARMUP_EPOCHS = 5
    
    FOCAL_WEIGHT = 1.0
    SUPCON_WEIGHT = 0.10
    SUPCON_TEMP = 0.07
    BASE_SMOOTHING = 0.05
    FOCAL_GAMMA = 2.0
    
    WEIGHT_DECAY = 1e-4
    DROPOUT_RATE = 0.2
    
    UNFREEZE_EPOCH = 5
    EARLY_STOPPING_PATIENCE = 15
    MIN_DELTA = 0.001
    
    CLASS_NAMES = ['Healthy', 'Ring', 'Trophozoite', 'Schizont', 'Gametocyte']
    NUM_CLASSES = 5
    
    STAGE_NAME_TO_ID = {'Healthy': 0, 'Ring': 1, 'Trophozoite': 2, 'Schizont': 3, 'Gametocyte': 4}
    
    DATASET_MEAN = [0.536918, 0.504134, 0.600036]
    DATASET_STD = [0.237723, 0.259878, 0.200624]
    
    DATASET_SPLITS = {
        'BBBC041-Test': {'train': 0.0, 'val': 0.0, 'test': 1.0},
        'BBBC041-Train': {'train': 0.9, 'val': 0.1, 'test': 0.0},
        'Plasmodium2019': {'train': 0.8, 'val': 0.1, 'test': 0.1},
        'IML': {'train': 0.8, 'val': 0.1, 'test': 0.1},
        'MP-IDB': {'train': 0.8, 'val': 0.1, 'test': 0.1},
        'Gambia': {'train': 0.8, 'val': 0.1, 'test': 0.1},
    }
    
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

config = Config()

print(f"\nMulticlass Classification: {config.CLASS_NAMES}")
print(f"Loss: Local Smoothing Focal + SupCon")
print(f"SupCon: τ={config.SUPCON_TEMP}, λ={config.SUPCON_WEIGHT}")


# AUGMENTATION LEVELS


In [ ]:
class GaussianNoise:
    """Add Gaussian noise to tensor.
    
    Args:
        p: Probability of applying noise (default: 0.3)
        sigma_range: Range for noise standard deviation (default: (0.0, 0.02))
    """
    
    def __init__(self, p=0.3, sigma_range=(0.0, 0.02)):
        self.p = p
        self.sigma_range = sigma_range
    
    def __call__(self, tensor):
        if random.random() < self.p:
            sigma = random.uniform(*self.sigma_range)
            noise = torch.randn_like(tensor) * sigma
            return torch.clamp(tensor + noise, 0, 1)
        return tensor

def get_light_transforms():
    """Get L1 light augmentation transforms.
    
    Returns:
        Composed transforms with light augmentation
    """
    return transforms.Compose([
        transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.90, 1.00)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(180, fill=128),
        transforms.ColorJitter(brightness=0.05, contrast=0.05),
        transforms.ToTensor(),
        transforms.Normalize(config.DATASET_MEAN, config.DATASET_STD),
    ])


def get_strong_transforms():
    """Get L2 strong augmentation transforms.
    
    Returns:
        Composed transforms with strong augmentation
    """
    return transforms.Compose([
        transforms.RandomResizedCrop(config.IMG_SIZE, scale=(0.90, 1.00)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(180, fill=128),
        transforms.ColorJitter(
            brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02
        ),
        transforms.RandomAdjustSharpness(sharpness_factor=2.0, p=0.3),
        transforms.ToTensor(),
        GaussianNoise(p=0.3, sigma_range=(0.0, 0.02)),
        transforms.Normalize(config.DATASET_MEAN, config.DATASET_STD),
    ])


def get_val_transforms():
    """Get validation transforms without augmentation.
    
    Returns:
        Composed transforms for validation/testing
    """
    return transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(config.DATASET_MEAN, config.DATASET_STD),
    ])


# DATA LOADING


In [ ]:
def discover_datasets(base_path):
    """Discover available datasets in the base path.
    
    Args:
        base_path: Base directory path containing datasets
        
    Returns:
        List of dataset names found
    """
    base = Path(base_path)
    found = []
    print("\nDiscovering datasets...")
    
    for subdir in sorted(base.iterdir()):
        if not subdir.is_dir():
            continue
            
        roi_cons = subdir / "ROI_consolidated"
        roi_dir = subdir / "ROI"
        json_dir = (
            roi_cons if roi_cons.exists() 
            else (roi_dir if roi_dir.exists() else None)
        )
        
        if not json_dir:
            continue
            
        json_files = list(json_dir.glob("*.json"))
        if json_files:
            print(f"  {subdir.name}: {len(json_files)} files")
            found.append(subdir.name)
            
    return found

def load_split(base_path, ds_name, split_type):
    """Load a specific split from a dataset.
    
    Args:
        base_path: Base directory path
        ds_name: Dataset name
        split_type: Type of split ('train', 'val', or 'test')
        
    Returns:
        List of data dictionaries
    """
    split_cfg = config.DATASET_SPLITS.get(
        ds_name, {'train': 0.8, 'val': 0.1, 'test': 0.1}
    )
    if split_cfg.get(split_type, 0) == 0:
        return []
    dpath = Path(base_path) / ds_name
    roi_cons = dpath / "ROI_consolidated"
    roi_dir = dpath / "ROI"
    json_dir = roi_cons if roi_cons.exists() else roi_dir
    if not json_dir.exists():
        return []
    img_groups = defaultdict(list)
    json_files = list(json_dir.glob("*.json"))
    for jf in tqdm(json_files, desc=f'  {ds_name}', leave=False):
        try:
            with open(jf, encoding='utf-8-sig') as f:
                data = json.load(f)
            rois = data.get('rois', [data])
            for roi in rois:
                rfn = roi.get('roi_filename')
                orig = roi.get('original_image', jf.stem)
                st = roi.get('stage')
                if not st or not isinstance(st, dict):
                    continue
                st_name = st.get('class_name')
                if not all([rfn, st_name]) or st_name not in config.STAGE_NAME_TO_ID:
                    continue
                img_path = None
                for check_dir in [roi_dir, roi_cons]:
                    if check_dir.exists():
                        for ext in ['', '.png']:
                            p = check_dir / (Path(rfn).stem + ext if ext else rfn)
                            if p.exists():
                                img_path = p
                                break
                    if img_path:
                        break
                if img_path:
                    cls_id = config.STAGE_NAME_TO_ID[st_name]
                    img_groups[orig].append({
                        'image_path': str(img_path),
                        'class_id': cls_id,
                        'original_stage': st_name,
                        'dataset': ds_name,
                    })
        except:
            pass
    imgs = sorted(list(img_groups.keys()))
    np.random.seed(42)
    np.random.shuffle(imgs)
    n = len(imgs)
    n_train = int(n * split_cfg['train'])
    n_val = int(n * split_cfg['val'])
    if split_type == 'train':
        sel = imgs[:n_train]
    elif split_type == 'val':
        sel = imgs[n_train:n_train + n_val]
    else:
        sel = imgs[n_train + n_val:]
    data = []
    for img in sel:
        data.extend(img_groups[img])
    return data

def load_all(base_path, split_type, datasets):
    """Load all data for a specific split across all datasets.
    
    Args:
        base_path: Base directory path
        split_type: Type of split ('train', 'val', or 'test')
        datasets: List of dataset names
        
    Returns:
        Tuple of (all_data, class_counts)
    """
    all_data = []
    print(f"\nLoading {split_type}:")
    for ds in datasets:
        if config.DATASET_SPLITS.get(ds, {}).get(split_type, 0) == 0:
            continue
        data = load_split(base_path, ds, split_type)
        if data:
            all_data.extend(data)
            print(f"  {ds}: {len(data)}")
    print(f"  Total: {len(all_data)}")
    class_counts = Counter([d['class_id'] for d in all_data])
    for cls_id in sorted(class_counts.keys()):
        print(f"    {config.CLASS_NAMES[cls_id]}: {class_counts[cls_id]}")
    return all_data, class_counts


# DYNAMIC HEALTHY ROTATION DATASET


In [ ]:
class BalancedMalariaDataset(Dataset):
    """Dataset with dynamic healthy image rotation per epoch.
    
    Args:
        healthy_data: List of healthy class samples
        other_data: List of non-healthy class samples
        transform: Image transforms to apply
        is_train: Whether this is training data (default: True)
        epoch: Current epoch number (default: 0)
    """
    
    def __init__(self, healthy_data, other_data, transform, is_train=True, epoch=0):
        self.healthy_data = healthy_data
        self.other_data = other_data
        self.transform = transform
        self.is_train = is_train
        self.epoch = epoch
        
        if is_train:
            self._resample_healthy()
        else:
            self.data = healthy_data + other_data
    
    def _resample_healthy(self):
        """Rotate through healthy images across epochs."""
        n_others = len(self.other_data)
        n_healthy = len(self.healthy_data)
        
        samples_per_epoch = min(n_others, n_healthy)
        offset = (self.epoch * samples_per_epoch) % n_healthy
        
        indices = [(offset + i) % n_healthy for i in range(samples_per_epoch)]
        selected_healthy = [self.healthy_data[i] for i in indices]
        
        self.data = selected_healthy + self.other_data
        random.shuffle(self.data)
        
        print(f"    Epoch {self.epoch}: Healthy={len(selected_healthy)}, Others={len(self.other_data)}")
    
    def set_epoch(self, epoch):
        self.epoch = epoch
        if self.is_train:
            self._resample_healthy()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        cls_id = item['class_id']
        
        try:
            img = Image.open(item['image_path']).convert('RGB')
        except:
            img = Image.new('RGB', (config.IMG_SIZE, config.IMG_SIZE), (128, 128, 128))
        
        img = self.transform(img)
        return img, cls_id


# MODEL WITH EMBEDDING OUTPUT


In [ ]:
class GeM(nn.Module):
    """Generalized Mean Pooling layer.
    
    Args:
        p: Power parameter (default: 3)
        eps: Small constant for numerical stability (default: 1e-6)
    """
    
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.adaptive_avg_pool2d(
            x.clamp(min=self.eps).pow(self.p), 1
        ).pow(1.0 / self.p).flatten(1)

class ConvNeXtV2Classifier(nn.Module):
    """ConvNeXt V2 classifier with embedding output support.
    
    Args:
        model_name: Name of the timm model
        num_classes: Number of output classes
        embedding_dim: Dimension of embedding layer (default: 512)
        dropout: Dropout rate (default: 0.2)
    """
    
    def __init__(self, model_name, num_classes, embedding_dim=512, dropout=0.2):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=True, num_classes=0, global_pool=''
        )
        self.in_features = self.backbone.num_features
        self.pool = GeM()
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(self.in_features, embedding_dim)
        self.bn = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)
        self.embedding_dim = embedding_dim

    def forward(self, x, return_embeddings=False):
        feat = self.backbone(x)
        pooled = self.pool(feat)
        pooled = self.drop(pooled)
        embeddings = F.relu(self.bn(self.fc(pooled)))
        logits = self.classifier(embeddings)
        
        if return_embeddings:
            return logits, embeddings
        return logits

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True


# SUPERVISED CONTRASTIVE LOSS


In [ ]:
class SupervisedContrastiveLoss(nn.Module):
    """Supervised Contrastive Loss.
    
    Args:
        temperature: Temperature parameter for scaling (default: 0.07)
    """
    
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, embeddings, labels):
        device = embeddings.device
        batch_size = embeddings.size(0)
        
        if batch_size < 2:
            return torch.tensor(0.0).to(device)
        
        embeddings = F.normalize(embeddings, p=2, dim=1)
        similarity_matrix = torch.matmul(embeddings, embeddings.T) / self.temperature
        
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)
        logits_mask = torch.ones_like(mask) - torch.eye(batch_size).to(device)
        mask = mask * logits_mask
        
        logits_max, _ = torch.max(similarity_matrix, dim=1, keepdim=True)
        logits = similarity_matrix - logits_max.detach()
        
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-12)
        
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, torch.ones_like(mask_pos_pairs), mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs
        
        loss = -mean_log_prob_pos.mean()
        return loss


# LOCAL SMOOTHING FOCAL LOSS (PER-CLASS ADAPTIVE)


In [ ]:
class LocalSmoothingFocalLoss(nn.Module):
    """Focal Loss with per-class local label smoothing.
    
    Local smoothing applies different smoothing values per class based on:
    - Minority classes get lower smoothing (preserve hard labels)
    - Majority classes get higher smoothing (regularize)
    """
    def __init__(self, class_counts, device, gamma=2.0, base_smoothing=0.05):
        super().__init__()
        counts = torch.FloatTensor(class_counts).to(device)
        self.log_class_prior = torch.log(counts / counts.sum())
        self.gamma = gamma
        self.num_classes = len(class_counts)
        self.device = device
        
        total = sum(class_counts)
        class_freq = [c / total for c in class_counts]
        
        max_freq = max(class_freq)
        self.local_smoothing = torch.FloatTensor([
            base_smoothing * (freq / max_freq) for freq in class_freq
        ]).to(device)
        
        print(f"   Local Smoothing Focal Loss: gamma={gamma}")
        for i, (cls_name, smooth) in enumerate(zip(config.CLASS_NAMES, self.local_smoothing.tolist())):
            print(f"     {cls_name}: smoothing={smooth:.4f}")
    
    def forward(self, logits, targets):
        adjusted_logits = logits + self.log_class_prior.unsqueeze(0)
        
        sample_smoothing = self.local_smoothing[targets]
        
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits)
            for i in range(logits.size(0)):
                s = sample_smoothing[i]
                smooth_targets[i].fill_(s / (self.num_classes - 1))
                smooth_targets[i, targets[i]] = 1.0 - s
        
        probs = F.softmax(adjusted_logits, dim=-1)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - pt) ** self.gamma
        
        log_probs = F.log_softmax(adjusted_logits, dim=-1)
        loss = -(smooth_targets * log_probs).sum(dim=-1)
        
        loss = (focal_weight * loss).mean()
        return loss


# COMBINED LOSS


In [ ]:
class CombinedLoss(nn.Module):
    """Combined Focal and Supervised Contrastive Loss.
    
    Args:
        class_counts: List of sample counts per class
        device: Torch device
        focal_weight: Weight for focal loss (default: 1.0)
        supcon_weight: Weight for contrastive loss (default: 0.10)
        gamma: Focal loss gamma parameter (default: 2.0)
        base_smoothing: Base smoothing value (default: 0.1)
        temperature: Temperature for contrastive loss (default: 0.07)
    """
    
    def __init__(self, class_counts, device, focal_weight=1.0, supcon_weight=0.10,
                 gamma=2.0, base_smoothing=0.1, temperature=0.07):
        super().__init__()
        
        self.focal_loss = LocalSmoothingFocalLoss(class_counts, device, gamma, base_smoothing)
        self.supcon_loss = SupervisedContrastiveLoss(temperature)
        
        self.focal_weight = focal_weight
        self.supcon_weight = supcon_weight
        
        print(f"   Combined Loss: Focal={focal_weight}, SupCon={supcon_weight} (τ={temperature})")
    
    def forward(self, logits, embeddings, labels):
        focal = self.focal_loss(logits, labels)
        supcon = self.supcon_loss(embeddings, labels)
        
        total = self.focal_weight * focal + self.supcon_weight * supcon
        
        return total, {
            'focal': focal.item(),
            'supcon': supcon.item(),
            'total': total.item()
        }


# SCHEDULER


In [ ]:
class WarmupCosineScheduler:
    """Learning rate scheduler with warmup and cosine annealing.
    
    Args:
        optimizer: PyTorch optimizer
        warmup_epochs: Number of warmup epochs
        total_epochs: Total number of training epochs
        min_lr: Minimum learning rate (default: 1e-6)
    """
    
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]
    
    def step(self, epoch):
        if epoch < self.warmup_epochs:
            factor = (epoch + 1) / self.warmup_epochs
            for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
                pg['lr'] = base_lr * factor
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
                pg['lr'] = self.min_lr + (base_lr - self.min_lr) * (1 + np.cos(np.pi * progress)) / 2
    
    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups]


# TRAINING


In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, scaler):
    """Train for one epoch.
    
    Args:
        model: Neural network model
        loader: Data loader
        criterion: Loss function
        optimizer: Optimizer
        device: Torch device
        scaler: Gradient scaler for mixed precision
        
    Returns:
        Dictionary of training metrics
    """
    model.train()
    total_loss = 0
    loss_components = {'focal': 0, 'supcon': 0}
    preds_all, labels_all = [], []
    
    for imgs, labels in tqdm(loader, desc='Train', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            logits, embeddings = model(imgs, return_embeddings=True)
            loss, loss_dict = criterion(logits, embeddings, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss_dict['total'] * imgs.size(0)
        for k in loss_components:
            loss_components[k] += loss_dict[k] * imgs.size(0)
        
        preds_all.extend(logits.argmax(1).cpu().numpy().tolist())
        labels_all.extend(labels.cpu().numpy().tolist())
    
    n = len(labels_all)
    acc = accuracy_score(labels_all, preds_all)
    return {
        'loss': total_loss / n,
        'accuracy': acc,
        'focal_loss': loss_components['focal'] / n,
        'supcon_loss': loss_components['supcon'] / n,
    }

@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validate the model.
    
    Args:
        model: Neural network model
        loader: Data loader
        criterion: Loss function
        device: Torch device
        
    Returns:
        Tuple of (metrics_dict, predictions, labels)
    """
    model.eval()
    total_loss = 0
    preds_all, labels_all = [], []
    
    for imgs, labels in tqdm(loader, desc='Val', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        
        with torch.cuda.amp.autocast():
            logits, embeddings = model(imgs, return_embeddings=True)
            loss, loss_dict = criterion(logits, embeddings, labels)
        
        total_loss += loss_dict['total'] * labels.size(0)
        preds_all.extend(logits.argmax(1).cpu().numpy().tolist())
        labels_all.extend(labels.cpu().numpy().tolist())
    
    acc = accuracy_score(labels_all, preds_all)
    precision = precision_score(labels_all, preds_all, average='macro', zero_division=0)
    recall = recall_score(labels_all, preds_all, average='macro', zero_division=0)
    f1 = f1_score(labels_all, preds_all, average='macro', zero_division=0)
    
    return {
        'loss': total_loss / len(labels_all),
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }, preds_all, labels_all


# VISUALIZATION


In [ ]:
def plot_confusion_matrix(labels, preds, class_names, title, save_path):
    """Plot and save confusion matrix.
    
    Args:
        labels: True labels
        preds: Predicted labels
        class_names: List of class names
        title: Plot title
        save_path: Path to save the figure
        
    Returns:
        Confusion matrix array
    """
    cm = confusion_matrix(labels, preds)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={'size': 20})
    axes[0].set_title('Counts', fontsize=14)
    axes[0].set_xlabel('Predicted', fontsize=12)
    axes[0].set_ylabel('Actual', fontsize=12)
    
    sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Greens', ax=axes[1],
                xticklabels=class_names, yticklabels=class_names,
                annot_kws={'size': 20})
    axes[1].set_title('Recall (Row-wise %)', fontsize=14)
    axes[1].set_xlabel('Predicted', fontsize=12)
    axes[1].set_ylabel('Actual', fontsize=12)
    
    plt.suptitle(title, fontweight='bold', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    return cm

def plot_metrics_bar(metrics, title, save_path):
    """Plot and save metrics bar chart.
    
    Args:
        metrics: Dictionary of metrics
        title: Plot title
        save_path: Path to save the figure
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    values = [
        metrics['accuracy'], metrics['precision'], 
        metrics['recall'], metrics['f1']
    ]
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
    
    bars = ax.bar(names, values, color=colors, edgecolor='black', linewidth=1.5)
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=14, fontweight='bold')
    
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title(title, fontsize=16, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

def plot_learning_curves(history, title, save_path):
    """Plot and save learning curves.
    
    Args:
        history: Training history dictionary
        title: Plot title
        save_path: Path to save the figure
    """
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
    axes[0, 0].plot(epochs, history['val_loss'], 'r-', label='Val', linewidth=2)
    axes[0, 0].set_title('Total Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    axes[0, 1].plot(epochs, history['train_acc'], 'b-', label='Train', linewidth=2)
    axes[0, 1].plot(epochs, history['val_acc'], 'r-', label='Val', linewidth=2)
    axes[0, 1].set_title('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    axes[1, 0].plot(epochs, history['focal_loss'], 'g-', label='Focal', linewidth=2)
    axes[1, 0].set_title('Focal Loss')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    axes[1, 1].plot(epochs, history['supcon_loss'], 'purple', label='SupCon', linewidth=2)
    axes[1, 1].set_title('SupCon Loss')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


# SINGLE RUN FUNCTION


In [ ]:
def run_experiment(aug_level, train_healthy, train_others, val_data, test_data, found_datasets):
    """Run a single experiment with specified augmentation level."""
    
    aug_name = "L1_Light" if aug_level == 1 else "L2_Strong"
    output_dir = f"{config.OUTPUT_BASE}/stronger_supcon_{aug_name}"
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f"RUN: {aug_name} Augmentation")
    print(f"SupCon: τ={config.SUPCON_TEMP}, λ={config.SUPCON_WEIGHT}")
    print(f"Output: {output_dir}")
    print(f"{'='*70}")
    
    set_seed(42)
    
    train_transform = get_light_transforms() if aug_level == 1 else get_strong_transforms()
    val_transform = get_val_transforms()
    
    train_ds = BalancedMalariaDataset(train_healthy, train_others, train_transform, is_train=True)
    
    val_healthy = [d for d in val_data if d['class_id'] == 0]
    val_others = [d for d in val_data if d['class_id'] != 0]
    val_ds = BalancedMalariaDataset(val_healthy, val_others, val_transform, is_train=False)
    
    test_healthy = [d for d in test_data if d['class_id'] == 0]
    test_others = [d for d in test_data if d['class_id'] != 0]
    test_ds = BalancedMalariaDataset(test_healthy, test_others, val_transform, is_train=False)
    
    others_counts = Counter([d['class_id'] for d in train_others])
    total_others = len(train_others)
    
    class_counts = [total_others]
    for i in range(1, 5):
        class_counts.append(others_counts.get(i, 0))
    
    train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True,
                              num_workers=config.NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                            num_workers=config.NUM_WORKERS, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                             num_workers=config.NUM_WORKERS, pin_memory=True)
    
    print(f"Loaders: Train={len(train_loader)}, Val={len(val_loader)}, Test={len(test_loader)}")
    
    model = ConvNeXtV2Classifier(
        config.MODEL_NAME, 
        config.NUM_CLASSES,
        embedding_dim=config.EMBEDDING_DIM,
        dropout=config.DROPOUT_RATE
    )
    model.freeze_backbone()
    model = model.to(config.DEVICE)
    
    criterion = CombinedLoss(
        class_counts=class_counts,
        device=config.DEVICE,
        focal_weight=config.FOCAL_WEIGHT,
        supcon_weight=config.SUPCON_WEIGHT,
        gamma=config.FOCAL_GAMMA,
        base_smoothing=config.BASE_SMOOTHING,
        temperature=config.SUPCON_TEMP
    )
    
    optimizer = optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': config.LEARNING_RATE * 0.1},
        {'params': model.pool.parameters(), 'lr': config.LEARNING_RATE},
        {'params': model.fc.parameters(), 'lr': config.LEARNING_RATE},
        {'params': model.bn.parameters(), 'lr': config.LEARNING_RATE},
        {'params': model.classifier.parameters(), 'lr': config.LEARNING_RATE},
    ], weight_decay=config.WEIGHT_DECAY)
    
    scheduler = WarmupCosineScheduler(optimizer, config.WARMUP_EPOCHS, config.EPOCHS)
    scaler = torch.cuda.amp.GradScaler()
    
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'val_f1': [], 'lr': [],
        'focal_loss': [], 'supcon_loss': [],
    }
    
    best_f1 = 0
    no_improve = 0
    
    print(f"\nTraining...")
    
    for epoch in range(config.EPOCHS):
        print(f"\nEpoch {epoch+1}/{config.EPOCHS}")
        
        train_ds.set_epoch(epoch)
        scheduler.step(epoch)
        history['lr'].append(scheduler.get_lr()[0])
        
        if epoch == config.UNFREEZE_EPOCH:
            model.unfreeze_backbone()
            print("Backbone unfrozen")
        
        train_m = train_epoch(model, train_loader, criterion, optimizer, config.DEVICE, scaler)
        val_m, _, _ = validate(model, val_loader, criterion, config.DEVICE)
        
        history['train_loss'].append(train_m['loss'])
        history['val_loss'].append(val_m['loss'])
        history['train_acc'].append(train_m['accuracy'])
        history['val_acc'].append(val_m['accuracy'])
        history['val_f1'].append(val_m['f1'])
        history['focal_loss'].append(train_m['focal_loss'])
        history['supcon_loss'].append(train_m['supcon_loss'])
        
        print(f"  Train: Loss={train_m['loss']:.4f} | Acc={train_m['accuracy']:.4f}")
        print(f"    Focal={train_m['focal_loss']:.4f} SupCon={train_m['supcon_loss']:.4f}")
        print(f"  Val:   Loss={val_m['loss']:.4f} | Acc={val_m['accuracy']:.4f} | F1={val_m['f1']:.4f}")
        
        if val_m['f1'] > best_f1 + config.MIN_DELTA:
            best_f1 = val_m['f1']
            no_improve = 0
            torch.save({
                'model': model.state_dict(),
                'best_f1': best_f1,
                'epoch': epoch,
            }, f"{output_dir}/best_model.pth")
            print(f"  Best F1: {best_f1:.4f}")
        else:
            no_improve += 1
        
        if no_improve >= config.EARLY_STOPPING_PATIENCE:
            print("Early stopping")
            break
    
    with open(f"{output_dir}/history.pkl", 'wb') as f:
        pickle.dump(history, f)
    with open(f"{output_dir}/history.json", 'w') as f:
        json.dump(history, f, indent=2)
    
    plot_learning_curves(history, f"Training Curves - Stronger SupCon + {aug_name}", 
                         f"{output_dir}/learning_curves.png")
    
    print(f"\nTEST")
    
    checkpoint = torch.load(f"{output_dir}/best_model.pth", weights_only=False)
    model.load_state_dict(checkpoint['model'])
    
    test_m, preds, labels = validate(model, test_loader, criterion, config.DEVICE)
    
    print(f"\nAccuracy:  {test_m['accuracy']:.4f}")
    print(f"Precision: {test_m['precision']:.4f}")
    print(f"Recall:    {test_m['recall']:.4f}")
    print(f"F1:        {test_m['f1']:.4f}")
    
    print("\n" + classification_report(labels, preds, target_names=config.CLASS_NAMES, digits=4))
    
    plot_confusion_matrix(labels, preds, config.CLASS_NAMES,
                          f"Stronger SupCon + {aug_name} - Test",
                          f"{output_dir}/confusion_matrix.png")
    
    plot_metrics_bar(test_m, f"Test Metrics - {aug_name}", f"{output_dir}/test_metrics.png")
    
    results = {
        'model': config.MODEL_NAME,
        'augmentation': aug_name,
        'supcon_params': {'temperature': config.SUPCON_TEMP, 'weight': config.SUPCON_WEIGHT},
        'test_metrics': test_m,
        'best_f1': best_f1,
    }
    
    with open(f"{output_dir}/results.json", 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\nSaved to: {output_dir}")
    
    return history, results


# MAIN - RUN AUGMENTATION EXPERIMENTS


In [ ]:
def main():
    """Run the main training pipeline.
    
    Returns:
        Dictionary of all experiment results
    """
    print("=" * 70)
    print("ConvNeXt V2 Multiclass Classification - Stronger SupCon")
    print(f"τ={config.SUPCON_TEMP}, λ={config.SUPCON_WEIGHT}")
    print("Running experiments: L1 Light Augmentation")
    print("=" * 70)
    print(f"Device: {config.DEVICE}")
    
    found_datasets = discover_datasets(config.BASE_PATH)
    if not found_datasets:
        raise RuntimeError("No datasets!")
    
    train_data, _ = load_all(config.BASE_PATH, 'train', found_datasets)
    val_data, _ = load_all(config.BASE_PATH, 'val', found_datasets)
    test_data, _ = load_all(config.BASE_PATH, 'test', found_datasets)
    
    train_healthy = [d for d in train_data if d['class_id'] == 0]
    train_others = [d for d in train_data if d['class_id'] != 0]
    
    print(f"\nTraining data:")
    print(f"   Healthy: {len(train_healthy)}")
    print(f"   Others: {len(train_others)}")
    
    all_results = {}
    
    h2, r2 = run_experiment(2, train_healthy, train_others, val_data, test_data, found_datasets)
    all_results['L2_Strong'] = r2
    
    print(f"\n{'='*70}")
    print("SUMMARY - Stronger SupCon (τ=0.07, λ=0.10)")
    print(f"{'='*70}")
    print(f"{'Augmentation':<15} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
    print("-"*55)
    for aug_name, res in all_results.items():
        m = res['test_metrics']
        print(f"{aug_name:<15} {m['accuracy']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f} {m['f1']:>10.4f}")
    
    with open(f"{config.OUTPUT_BASE}/stronger_supcon_summary.json", 'w') as f:
        json.dump(all_results, f, indent=2)
    
    print(f"\nAll experiments complete!")
    
    return all_results

if __name__ == "__main__":
    all_results = main()
